In [45]:
!pip install transformers datasets torch
import torch
from datasets import Dataset
from google.colab import drive
import ast
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_fscore_support, classification_report
from transformers import AutoModelForTokenClassification, AutoTokenizer
from torch.nn.functional import softmax

In [2]:
drive.mount('/content/drive')
tfmPath = '/content/drive/MyDrive/TFM'
trainingFilesPath = tfmPath+'/archivos_entrenamiento'
validationFilePath = tfmPath+'/archivos_validacion'
testFilePath = tfmPath+'/archivo_test'
model_path = tfmPath+'/modeloBert'

Mounted at /content/drive


In [3]:
archivo = trainingFilesPath+'/train.txt'

with open(archivo, 'r', encoding='utf-8') as file:
    content = file.read()
    data = ast.literal_eval(content)

In [ ]:
print(data)

[('Finalmente llegué a Gómez Palacio y visité las instalaciones de Bellas Artes, conocí a los que iban a ser mis alumnos. Temblaba todo el tiempo pese al calor que hacía.', {'entities': [(110, 117, 'alu')]}), ('Cuando quise dejar la fila el japonés se abalanzó sobre mí blandiendo un bastón de madera, el mismo con el que golpeaba a los alumnos que así se lo pedían. Es decir, Ejo ofrecía el bastón, los alumnos decían sí o no y en caso de ser la respuesta afirmativa Ejo les descerrajaba unos planazos que atronaban el espacio en penumbra impregnado de incienso. ', {'entities': [(126, 133, 'alu'), (193, 200, 'alu')]}), ('El que el director encargue a los alumnos más aventajados para que sirvan de monitores para orientar y ayudar a los de menor rendimiento, puede ayudarle y apoyar el trabajo del director.', {'entities': [(34, 41, 'alu')]}), ('De allí que también pueda insertarse, en estas páginas, la evocación de una que otra visita guiada, realizadas expresamente para mis alumnos, cuando as

In [6]:
archivo_val = validationFilePath+'/val.txt'

with open(archivo_val, 'r', encoding='utf-8') as file:
    content = file.read()
    data_val = ast.literal_eval(content)


In [ ]:
print(data_val)

[('Les corresponde, pues, a maestros y profesores la tarea de organizar las actividades de fomento y disfrute de sus alumnos con la lectura, si bien este compromiso debe ser asumido por toda la comunidad educativa. ', {'entities': [(36, 46, 'profs'), (114, 121, 'alu')]}), ('No es lo mismo partir de cero, con alumnos que no han tenido contacto alguno con libros, que empezar con niños cargados de prejuicios negativos por haber vivido experiencias incómodas en su acercamiento a los libros; o trabajar con otros que aprecian y aman la lectura.', {'entities': [(35, 42, 'alu')]}), ('Por lo demás, esperamos que el tipo de iniciativas y propuestas, a las que vamos a hacer referencia a continuación, vayan depositando en los alumnos unos sedimentos que les permitan construir día a día el hábito lector.', {'entities': [(141, 148, 'alu')]}), ('La biblioteca del centro tiene que ser para los alumnos un espacio acogedor. ', {'entities': [(48, 55, 'alu')]}), ('Este puede ser un momento especial que al

In [7]:
archivo_test = testFilePath+'/test.txt'

with open(archivo_test, 'r', encoding='utf-8') as file:
    content = file.read()
    data_test = ast.literal_eval(content)

In [ ]:
print(data_test)

[('Las expectativas se han cubierto satisfactoriamente pues todos los alumnos que pasan de la tercera actividad planteada suelen concluir el curso con éxito.', {'entities': [(67, 74, 'alu')]}), ('Y un bajísimo nivel en habilidades instrumentales (lectoescritura y cálculo matemático, básicamente) en los alumnos que superan esa etapa e incluso el bachillerato. Causa del problema: el actual sistema educativo.', {'entities': [(108, 115, 'alu')]}), ('Información Geográfica 11-12 Figura 23. Grupos de docentes de 2011-2, por programa académico, matriculados como alumnos en el Aula Virtual desde febrero de 2011 hasta mayo de 2012.', {'entities': [(112, 119, 'alu')]}), ('Por ello hicimos grupos en os que había unos 3 alumnos de cada centro. Hemos realizado una gymkhana por Cangas de Onis y Covadonga en la que tuvimos que descubrir cosas.', {'entities': []}), ('Los alumnos también tienen una opinión muy positiva de esta herramienta ya que les facilita el seguimiento de la asignatura. Un nivel de

In [8]:
# Preprocesamiento de los datos

tokenizer = BertTokenizerFast.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')

def preprocess_data(data):
    tokenized_data = []

    for text, entity_dict in data:
        entities = entity_dict['entities']

        # Se tokeniza el texto
        tokenized_inputs = tokenizer(text, truncation=True, padding='max_length', max_length=128, is_split_into_words=False)

        # Se inicializa la lista de etiquetas con -100 para ignorar
        labels = [-100] * len(tokenized_inputs["input_ids"])

        # Se asignan etiquetas a los tokens de las entidades
        for start, end, entity_label in entities:
            # Se tokeniza la parte del texto que corresponde a la entidad
            entity_tokens = tokenizer.encode(text[start:end], add_special_tokens=False)

            # Se busca la posición de los tokens de la entidad en `tokenized_inputs`
            for idx in range(len(tokenized_inputs["input_ids"])):
                # Se verifica si el segmento de tokens coincide
                if tokenized_inputs["input_ids"][idx:idx + len(entity_tokens)] == entity_tokens:
                    entity_id = label2id[entity_label]
                    for i in range(len(entity_tokens)):
                        labels[idx + i] = entity_id  # Se asigna la etiqueta a todos los subtokens
                    break

        # Se añaden las etiquetas a la entrada tokenizada
        tokenized_inputs["labels"] = labels
        tokenized_data.append(tokenized_inputs)

    return tokenized_data


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

In [9]:
#Configuración del Modelo BERT para NER

# Se recopilan las etiquetas únicas de los datos
unique_labels = set()
for _, entity_dict in data:
    for _, _, label in entity_dict['entities']:
        unique_labels.add(label)

# Se converte el conjunto en una lista ordenada para asegurar consistencia
label_list = sorted(list(unique_labels))

# Se crea un id2label y un label2id dinámicamente
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in id2label.items()}

# Se imprimen los resultados para verificar
print("id2label:", id2label)
print("label2id:", label2id)


id2label: {0: 'alu', 1: 'bus', 2: 'cam', 3: 'cl', 4: 'dis1', 5: 'dis2', 6: 'dis3', 7: 'dis4', 8: 'dis5', 9: 'dis6', 10: 'hab', 11: 'ing', 12: 'inv', 13: 'profs', 14: 'vdis1', 15: 'vdis2'}
label2id: {'alu': 0, 'bus': 1, 'cam': 2, 'cl': 3, 'dis1': 4, 'dis2': 5, 'dis3': 6, 'dis4': 7, 'dis5': 8, 'dis6': 9, 'hab': 10, 'ing': 11, 'inv': 12, 'profs': 13, 'vdis1': 14, 'vdis2': 15}


In [ ]:
# Se carga el modelo BERT
model = BertForTokenClassification.from_pretrained('dccuchile/bert-base-spanish-wwm-cased', num_labels=len(id2label), id2label=id2label, label2id=label2id)

# Se configura el entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.001,
    report_to="none"
)



pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
#Datos de train
df = preprocess_data(data)

# Convertimos a Dataset de Hugging Face
df = pd.DataFrame(df)
df_subset = df.iloc[:2136]
dataset_train = Dataset.from_pandas(df_subset)


In [11]:
#Se hace un print para verificar si está en el formato adecuado
print(type(dataset_train))

<class 'datasets.arrow_dataset.Dataset'>


In [12]:
#Datos de validation

df_val = preprocess_data(data_val)

# Convertimos a Dataset de Hugging Face
df_val = pd.DataFrame(df_val)
df_val_subset = df.iloc[:403]
dataset_val = Dataset.from_pandas(df_val_subset)


In [13]:
#Se hace un print para verificar si está en el formato adecuado
print(type(dataset_val))

<class 'datasets.arrow_dataset.Dataset'>


In [14]:
#Datos de train
df_test = preprocess_data(data_test)

# Convertimos a Dataset de Hugging Face
df_test = pd.DataFrame(df_test)
df_subset_test = df_test.iloc[:443]
dataset_test = Dataset.from_pandas(df_subset_test)


In [15]:
dataset = DatasetDict({
    "train": dataset_train,
    "validation": dataset_val,
    "test": dataset_test
})

In [ ]:
#Se hace un print para verificar si está en el formato adecuado
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['attention_mask', 'input_ids', 'labels', 'token_type_ids'],
        num_rows: 2136
    })
    validation: Dataset({
        features: ['attention_mask', 'input_ids', 'labels', 'token_type_ids'],
        num_rows: 403
    })
    test: Dataset({
        features: ['attention_mask', 'input_ids', 'labels', 'token_type_ids'],
        num_rows: 407
    })
})


In [ ]:
# Entrenamiento

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
)

trainer.train()
results = trainer.evaluate()
print(results)

<ipython-input-21-d360694a8063>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.005300,0.002462
2,0.001700,0.000850
3,0.001200,0.000684


{'eval_loss': 0.0006837293040007353, 'eval_runtime': 185.9365, 'eval_samples_per_second': 2.167, 'eval_steps_per_second': 0.274, 'epoch': 3.0}


In [ ]:
# Se guardar el modelo entrenado en un directorio
trainer.save_model(tfmPath+'/modeloBert')

In [ ]:
# Guardar el modelo y el tokenizador
model.save_pretrained(tfmPath+'/modeloBert')
tokenizer.save_pretrained(tfmPath+'/modeloBert')

('/content/drive/MyDrive/TFM/modeloBert/tokenizer_config.json',
 '/content/drive/MyDrive/TFM/modeloBert/special_tokens_map.json',
 '/content/drive/MyDrive/TFM/modeloBert/vocab.txt',
 '/content/drive/MyDrive/TFM/modeloBert/added_tokens.json',
 '/content/drive/MyDrive/TFM/modeloBert/tokenizer.json')

In [4]:
# Cargar el modelo y el tokenizador
model = AutoModelForTokenClassification.from_pretrained(tfmPath+'/modeloBert')
tokenizer = AutoTokenizer.from_pretrained(tfmPath+'/modeloBert')

In [17]:
# Se extraen las etiquetas del modelo
id2label = model.config.id2label  # Diccionario {id: etiqueta}
label2id = model.config.label2id  # Diccionario {etiqueta: id}

# Se converte el id2label a lista
label_list = [label for _, label in sorted(id2label.items())]
print(label_list)

['alu', 'bus', 'cam', 'cl', 'dis1', 'dis2', 'dis3', 'dis4', 'dis5', 'dis6', 'hab', 'ing', 'inv', 'profs', 'vdis1', 'vdis2']


In [46]:
# Se evalua el modelo con dato previamente no vistos
def evaluar_modelo(model, test_dataset):

    y_true = []
    y_pred = []

    #Se itera sobre los datos del conjunto de prueba
    for example in test_dataset:
        # Se preparan las entradas para el modelo
        inputs = {
            "input_ids": torch.tensor([example['input_ids']]),
            "attention_mask": torch.tensor([example['attention_mask']])
        }
        if "token_type_ids" in example:
            inputs["token_type_ids"] = torch.tensor([example["token_type_ids"]])

        # Se obtenen las etiquetas reales
        labels = example['labels']

        # Se realiza la predicción
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits
        predictions = logits.argmax(dim=-1).squeeze().tolist()

        # Se alinear etiquetas reales con las predicciones (ignorar padding)
        aligned_labels = []
        aligned_predictions = []
        for label, prediction in zip(labels, predictions):
            if label != -100:  # Ignorar tokens de padding
                aligned_labels.append(label)
                aligned_predictions.append(prediction)

        y_true.extend(aligned_labels)
        y_pred.extend(aligned_predictions)

    #Se calcular métricas globales
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', labels=np.unique(y_true)
    )
    f1_micro = precision_recall_fscore_support(y_true, y_pred, average='micro')[2]
    f1_macro = precision_recall_fscore_support(y_true, y_pred, average='macro')[2]

    #Se crea un informe detallado
    reporte = classification_report(y_true, y_pred, digits=4, zero_division=0)

    #Se imprimir el informe y métricas
    print("=== Informe ===")
    print(reporte)
    print(f"F1 Micro: {f1_micro:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "reporte": reporte
    }


In [47]:
evaluar_modelo(model, dataset_test)

=== Informe ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        26
           1     1.0000    1.0000    1.0000        63
           2     1.0000    1.0000    1.0000        34
           3     1.0000    1.0000    1.0000        20
           4     1.0000    1.0000    1.0000        38
           5     1.0000    1.0000    1.0000        52
           6     1.0000    1.0000    1.0000        42
           7     1.0000    1.0000    1.0000        12
           8     1.0000    1.0000    1.0000        33
           9     1.0000    1.0000    1.0000        85
          10     1.0000    1.0000    1.0000        15
          11     1.0000    1.0000    1.0000        17
          12     1.0000    1.0000    1.0000        18
          13     1.0000    1.0000    1.0000        23
          14     1.0000    1.0000    1.0000        46
          15     1.0000    1.0000    1.0000        63

    accuracy                         1.0000       587
   macro a

{'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'f1_micro': 1.0,
 'f1_macro': 1.0,
 'reporte': '              precision    recall  f1-score   support\n\n           0     1.0000    1.0000    1.0000        26\n           1     1.0000    1.0000    1.0000        63\n           2     1.0000    1.0000    1.0000        34\n           3     1.0000    1.0000    1.0000        20\n           4     1.0000    1.0000    1.0000        38\n           5     1.0000    1.0000    1.0000        52\n           6     1.0000    1.0000    1.0000        42\n           7     1.0000    1.0000    1.0000        12\n           8     1.0000    1.0000    1.0000        33\n           9     1.0000    1.0000    1.0000        85\n          10     1.0000    1.0000    1.0000        15\n          11     1.0000    1.0000    1.0000        17\n          12     1.0000    1.0000    1.0000        18\n          13     1.0000    1.0000    1.0000        23\n          14     1.0000    1.0000    1.0000        46\n          15     1.000

In [ ]:
# Se pide un texto al importar al usuario
texto_input = input("Por favor, introduce el texto: ")

Por favor, introduce el texto: En el ámbito de la atención sanitaria, es fundamental brindar apoyo adecuado a quienes enfrentan diversas condiciones de salud. Por ejemplo, el discapacitado puede requerir acceso a terapias de rehabilitación especializadas, mientras que el minusválido a menudo necesita adaptaciones en su entorno para mejorar su movilidad y calidad de vida. De manera similar, el inválido puede beneficiarse de dispositivos de asistencia tecnológica que le permitan realizar tareas cotidianas de forma más independiente.  Existen condiciones más específicas, como la de un tetrapléjico, que generalmente requiere cuidados médicos continuos y soporte especializado para manejar su situación. Además, quienes padecen discapacidad a veces deben enfrentar barreras en el acceso a la educación y el empleo, lo cual limita sus oportunidades de desarrollo personal y profesional.  Por otro lado, algunos individuos que sufren discapacidad debido a accidentes o enfermedades pueden experiment

In [ ]:
# Se tokeniza el texto importado

def preprocess_text(texto_input, tokenizer):
    tokens = tokenizer(texto_input, return_tensors="pt", truncation=True, padding=True)
    return tokens


In [ ]:
# Se utiliza el modelo preentrando para hacer predicciones sobre las etiquetas de los tokens

def predict_entities(text, model, tokenizer, id2label):
    model.eval()  # Poner el modelo en modo evaluación

    # Se tokeniza el texto
    tokens = preprocess_text(text, tokenizer)

    # Se obtienen las predicciones del modelo
    with torch.no_grad():
        outputs = model(**tokens)
        logits = outputs.logits
    # Se calculan las probabilidades
    probabilities = softmax(logits, dim=-1)

    # Se define umbral
    confidence_threshold = 0.99

    # Se filtran las predicciones
    predictions = []
    confidences = []

    for token_probs in probabilities.squeeze(0):  # Iterar sobre tokens
        max_prob, predicted_class = torch.max(token_probs, dim=-1)
        if max_prob >= confidence_threshold:
            predictions.append(predicted_class.item())
            confidences.append(max_prob.item())
        else:
            predictions.append(None)
            confidences.append(None)

    # Se filtra con la attention mask
    attention_mask = tokens['attention_mask']
    filtered_predictions = [pred if mask == 1 else None for pred, mask in zip(predictions, attention_mask.squeeze().tolist())]

    # Se convierten los tokens en texto legible
    tokens_list = tokenizer.convert_ids_to_tokens(tokens["input_ids"].squeeze().tolist())

    # Se alinean las etiquetas ignorando tokens irrelevantes
    aligned_preds = []
    aligned_tokens = []

    for token, pred, mask in zip(tokens_list, filtered_predictions, tokens["attention_mask"].squeeze().tolist()):
        if mask == 1 and token not in ["[CLS]", "[SEP]"] and pred != None:
            aligned_preds.append(id2label.get(pred, "O"))  # Default "O" si predicción no está en id2label
            aligned_tokens.append(token)

    # Se crean la lista final de resultados
    result = list(zip(aligned_tokens, aligned_preds))

    return result



In [ ]:

result = predict_entities(texto_input, model, tokenizer, id2label)

# Se muestran resultados
for token, label in result:
    print(f"{token}: {label}")

discapaci: dis1
##tado: dis1
minusvá: dis2
##lido: dis2
inv: dis3
##ál: dis3
##ido: dis3
tetra: dis6
##pl: dis6
##éj: dis6
##ico: dis6
padecen: vdis2
discapacidad: vdis2
sufren: vdis1
discapacidad: vdis1
retrasado: dis5


In [ ]:
def join_subtokens_with_labels(result):
    # Lista para almacenar los resultados en el formato deseado
    joined_result = []

    # La primera palabra no tiene "##", así que se agrega directamente
    current_token, current_label = result[0]

    # Iteramos sobre los tokens y sus etiquetas
    for token, label in result[1:]:
        if token.startswith("##"):
            # Si el token empieza con "##", se une al anterior
            current_token += token[2:]
        else:
            joined_result.append((current_token, current_label))
            current_token, current_label = token, label

    # Se añade el último token y etiqueta
    joined_result.append((current_token, current_label))

    return joined_result

# Se unen los subtokens y las etiquetas
joined_result = join_subtokens_with_labels(result)

# Se imprimen los resultados
for token, label in joined_result:
    print(f"Token: {token}, Label: {label}")

Token: discapacitado, Label: dis1
Token: minusválido, Label: dis2
Token: inválido, Label: dis3
Token: tetrapléjico, Label: dis6
Token: padecen, Label: vdis2
Token: discapacidad, Label: vdis2
Token: sufren, Label: vdis1
Token: discapacidad, Label: vdis1
Token: retrasado, Label: dis5


In [ ]:
for token, label in result:
  if label == "alu":
    print(f"Considere cambiar 'los alumnos' por 'el alumnado'. ")
  if label == "profs":
    print(f"Considere cambiar 'los profesores' por 'el profesorado'. ")
  if label == "inv":
    print(f"Considere cambiar 'los investigadores' por 'el equipo de investigación'. ")
  if label == "hab":
    print(f"Considere cambiar 'los habitantes' por 'la población'. ")
  if label == "cam":
    print(f"Considere cambiar 'los camareros' por 'el personal de servicio'. ")
  if label == "ing":
    print(f"Considere cambiar 'los ingenieríeros' por 'el equipo de ingeniería'. ")
  if label == "bus":
    print(f"Considere cambiar 'los hombres de negocios' por 'las personas de negocios'. ")
  if label == "cl":
    print(f"Considere cambiar 'los clientes' por 'la clientela' ")
  if label == "dis1":
    print(f"Considere cambiar 'el discapacitado' por '(persona) con discapacidad'.")
  if label == "dis2":
    print(f"Considere cambiar 'el minusválido' por '(persona) con discapacidad'.")
  if label == "dis3":
    print(f"Considere cambiar 'el inválido' por '(persona) con discapacidad'.")
  if label == "dis4":
    print(f"Considere cambiar 'el disminuido' por '(persona) con discapacidad'.")
  if label == "dis5":
    print(f"Considere cambiar 'el retrasado' por '(persona) con discapacidad'.")
  if label == "dis6":
    print(f"Considere cambiar 'el tetrapléjico'  por 'la persona con tetrapléjia'.")
  if label == "vdis1":
    print(f"Considere cambiar 'sufir discapacidad' por 'tener discapacidad' ")
  if label == "vdis2":
    print(f"Considere cambiar 'padecer discapacidad' por 'tener discapacidad' ")
  if label == "":
    print(f"(No se han encontrado mejoras según los criterios establecidos.")

Considere cambiar 'el discapacitado' por '(persona) con discapacidad'.
Considere cambiar 'el discapacitado' por '(persona) con discapacidad'.
Considere cambiar 'el minusválido' por '(persona) con discapacidad'.
Considere cambiar 'el minusválido' por '(persona) con discapacidad'.
Considere cambiar 'el inválido' por '(persona) con discapacidad'.
Considere cambiar 'el inválido' por '(persona) con discapacidad'.
Considere cambiar 'el inválido' por '(persona) con discapacidad'.
Considere cambiar 'el tetrapléjico'  por 'la persona con tetrapléjia'.
Considere cambiar 'el tetrapléjico'  por 'la persona con tetrapléjia'.
Considere cambiar 'el tetrapléjico'  por 'la persona con tetrapléjia'.
Considere cambiar 'el tetrapléjico'  por 'la persona con tetrapléjia'.
Considere cambiar 'padecer discapacidad' por 'tener discapacidad' 
Considere cambiar 'padecer discapacidad' por 'tener discapacidad' 
Considere cambiar 'sufir discapacidad' por 'tener discapacidad' 
Considere cambiar 'sufir discapacidad' 